<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-Tuan/AIO-Conquer/blob/MinhKhoa/communities%26crime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Boruta -q

import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from boruta import BorutaPy
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LassoCV

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 3.5 MB/s eta 0:00:00


In [ ]:
drive.mount('/content/drive')
FILE_PATH = '/content/drive/MyDrive/conquer1/AIO-Conquer-Data/Data/dataset/communities_processed.csv'
try:
    df = pd.read_csv(FILE_PATH)
    print(f"Đã tải dữ liệu thành công! Kích thước: {df.shape}")
    df.info()
    df = df.copy()
except Exception as e:
    print(f"Lỗi: Không tìm thấy file hoặc đường dẫn sai. Chi tiết: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã tải dữ liệu thành công! Kích thước: (1994, 101)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1994 entries, 0 to 1993
Columns: 101 entries, population to ViolentCrimesPerPop
dtypes: float64(101)
memory usage: 1.5 MB


In [ ]:
df.head(5)

,population,householdsize,racepctblack,racePctWhite,racePctAsian,racePctHisp,agePct12t21,agePct12t29,agePct16t24,agePct65up,...,PctForeignBorn,PctBornSameState,PctSameHouse85,PctSameCity85,PctSameState85,LandArea,PopDens,PctUsePubTrans,LemasPctOfficDrugUn,ViolentCrimesPerPop
0,0.19,0.33,0.02,0.90,0.12,0.17,0.34,0.47,0.29,0.32,...,0.12,0.42,0.50,0.51,0.64,0.12,0.26,0.20,0.32,0.20
1,0.00,0.16,0.12,0.74,0.45,0.07,0.26,0.59,0.35,0.27,...,0.21,0.50,0.34,0.60,0.52,0.02,0.12,0.45,0.00,0.67
2,0.00,0.42,0.49,0.56,0.17,0.04,0.39,0.47,0.28,0.32,...,0.14,0.49,0.54,0.67,0.56,0.01,0.21,0.02,0.00,0.43
3,0.04,0.77,1.00,0.08,0.12,0.10,0.51,0.50,0.34,0.21,...,0.19,0.30,0.73,0.64,0.65,0.02,0.39,0.28,0.00,0.12
4,0.01,0.55,0.02,0.95,0.09,0.05,0.38,0.38,0.23,0.36,...,0.11,0.72,0.64,0.61,0.53,0.04,0.09,0.02,0.00,0.03


In [ ]:
target_colum = 'ViolentCrimesPerPop'
X = df.drop(columns=[target_colum,'BMI'], errors='ignore')
y = df[target_colum]

# 2. Chia tập Train/Test trước để tránh data leakage khi tính trung bình giá
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
lr_model = LinearRegression()

# 2. Huấn luyện mô hình trên tập dữ liệu đã encode
lr_model.fit(X_train, y_train)

# 3. Tiến hành dự đoán cân nặng
preds = lr_model.predict(X_test)

# 4. Đánh giá mô hình bằng R2-score, RMSE, MAE
r2_score_all = round(r2_score(y_test, preds), 3)
rmse_all = round(np.sqrt(mean_squared_error(y_test, preds)), 3)
mae_all = round(mean_absolute_error(y_test, preds), 3)

print(f"Chỉ số R2-score khi sử dụng Linear Regression: {r2_score_all}")
print(f"Chỉ số RMSE khi sử dụng Linear Regression: {rmse_all}")
print(f"Chỉ số MAE khi sử dụng Linear Regression: {mae_all}")

Chỉ số R2-score khi sử dụng Linear Regression: 0.638
Chỉ số RMSE khi sử dụng Linear Regression: 0.132
Chỉ số MAE khi sử dụng Linear Regression: 0.094


# **1.Boruta**

In [ ]:
# --- 1. Cấu hình và chạy Boruta ---
# BorutaPy yêu cầu mảng NumPy (.values) để tránh lỗi index
X_train_np = X_train.values
y_train_np = y_train.values

# Giữ nguyên Random Forest làm mô hình nền để Boruta tính toán feature_importances_
rf = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)
boruta_selector = BorutaPy(estimator=rf, n_estimators='auto', verbose=0, random_state=42, max_iter=100)
boruta_selector.fit(X_train_np, y_train_np)

# --- 2. Lọc và huấn luyện lại mô hình ---
# Lọc lấy các tính năng quan trọng được Boruta xác nhận
X_train_filtered = boruta_selector.transform(X_train_np)
X_test_filtered = boruta_selector.transform(X_test.values)

#Khởi tạo và huấn luyện mô hình Linear Regression trên tập dữ liệu đã lọc
lr_final = LinearRegression()
lr_final.fit(X_train_filtered, y_train_np)

# --- 3. Dự đoán và đánh giá điểm R2 ---
# Sử dụng mô hình Linear Regression vừa train để dự đoán
preds_boruta = lr_final.predict(X_test_filtered)
r2_score_boruta = round(r2_score(y_test.values, preds_boruta), 3)
rsme_boruta = round(np.sqrt(mean_squared_error(y_test.values, preds_boruta)), 3)
mae_boruta = round(mean_absolute_error(y_test.values, preds_boruta), 3)

# --- 4. Trích xuất danh sách tính năng được chọn ---
feature_names = X_train.columns
confirmed_features = feature_names[boruta_selector.support_].tolist()

# --- 5. Xuất kết quả báo cáo ---
print(f"{'KẾT QUẢ SÀN LỌC TÍNH NĂNG & HUẤN LUYỆN LINEAR REGRESSION':^60}")
print(f"• Chỉ số R2-score (Linear Regression) : {r2_score_boruta}")
print(f"• Chỉ số RMSE (Linear Regression)    : {rsme_boruta}")
print(f"• Chỉ số MAE (Linear Regression)     : {mae_boruta}")
print(f"• Số lượng tính năng được giữ lại   : {len(confirmed_features)}/{len(feature_names)}")
print(f"• Các tính năng được giữ lại:\n  {confirmed_features}")
print("="*60)

  KẾT QUẢ SÀN LỌC TÍNH NĂNG & HUẤN LUYỆN LINEAR REGRESSION  
• Chỉ số R2-score (Linear Regression) : 0.619
• Chỉ số RMSE (Linear Regression)    : 0.135
• Chỉ số MAE (Linear Regression)     : 0.096
• Số lượng tính năng được giữ lại   : 18/100
• Các tính năng được giữ lại:
  ['racepctblack', 'racePctWhite', 'pctWInvInc', 'PctPopUnderPov', 'MalePctDivorce', 'FemalePctDiv', 'TotalPctDiv', 'PctFam2Par', 'PctKids2Par', 'NumIlleg', 'PctIlleg', 'PctSpeakEnglOnly', 'PctLargHouseFam', 'PctPersDenseHous', 'PctHousLess3BR', 'HousVacant', 'NumStreet', 'PctUsePubTrans']


# **2.LASSO**

In [ ]:
# --- 1. Chuẩn hóa dữ liệu (Feature Scaling) ---
# Bước này bắt buộc phải có vì Lasso rất nhạy cảm với scale của dữ liệu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 2. Khởi tạo và chạy LassoCV để SÀNG LỌC TÍNH NĂNG ---
#tăng số vòng lặp lên vì mô hình chưa hội tụ
lasso = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, random_state=42,max_iter=10000, n_jobs=-1)
lasso.fit(X_train_scaled, y_train)

# Trích xuất danh sách tên các tính năng được Lasso giữ lại (Coef != 0)
feature_names = X_train.columns
kept_features = feature_names[lasso.coef_ != 0].tolist()
dropped_features = feature_names[lasso.coef_ == 0].tolist()

# --- 3. Lọc lại dữ liệu theo các tính năng được chọn ---
# Lấy vị trí index của các tính năng được giữ lại để lọc trên mảng numpy đã scaled
kept_features_idx = [X_train.columns.get_loc(col) for col in kept_features]
X_train_filtered = X_train_scaled[:, kept_features_idx]
X_test_filtered = X_test_scaled[:, kept_features_idx]

# --- 4. Huấn luyện mô hình LINEAR REGRESSION trên các tính năng đã lọc ---
lr_final = LinearRegression(n_jobs=-1)
lr_final.fit(X_train_filtered, y_train)

# --- 5. Dự đoán và tính toán các chỉ số đánh giá ---
preds_lr = lr_final.predict(X_test_filtered)
r2_score_lr = round(r2_score(y_test, preds_lr), 3)
rmse_lr = round(np.sqrt(mean_squared_error(y_test, preds_lr)), 3)
mae_lr = round(mean_absolute_error(y_test, preds_lr), 3)

# --- 6. Xuất kết quả báo cáo ---
print("="*60)
print(f"{'KẾT QUẢ SÀN LỌC BẰNG LASSO & HUẤN LUYỆN LINEAR REGRESSION':^60}")
print("="*60)
print(f"• Alpha tối ưu của Lasso             : {round(lasso.alpha_, 5)}")
print(f"• Chỉ số R2-score (Linear Regression) : {r2_score_lr}")
print(f"• Chỉ số RMSE (Linear Regression)    : {rmse_lr}")
print(f"• Chỉ số MAE (Linear Regression)     : {mae_lr}")
print(f"• Số lượng tính năng bị loại bỏ      : {len(dropped_features)}/{len(feature_names)}")
print(f"• Số lượng tính năng được giữ lại    : {len(kept_features)}/{len(feature_names)}")
print("-"*60)
if dropped_features:
    print(f"• Các tính năng bị loại bỏ (Lasso Coef = 0):\n  {dropped_features}\n")
print(f"• Các tính năng được chọn để chạy Linear Regression:\n  {kept_features}")
print("="*60)

 KẾT QUẢ SÀN LỌC BẰNG LASSO & HUẤN LUYỆN LINEAR REGRESSION  
• Alpha tối ưu của Lasso             : 0.00023
• Chỉ số R2-score (Linear Regression) : 0.638
• Chỉ số RMSE (Linear Regression)    : 0.132
• Chỉ số MAE (Linear Regression)     : 0.094
• Số lượng tính năng bị loại bỏ      : 18/100
• Số lượng tính năng được giữ lại    : 82/100
------------------------------------------------------------
• Các tính năng bị loại bỏ (Lasso Coef = 0):
  ['population', 'agePct12t21', 'agePct16t24', 'medIncome', 'perCapInc', 'TotalPctDiv', 'PersPerFam', 'PctImmigRec8', 'PctImmigRec10', 'PctRecImmig5', 'PctRecImmig10', 'PctLargHouseFam', 'PctHousOwnOcc', 'MedYrHousBuilt', 'OwnOccMedVal', 'RentMedian', 'PctSameHouse85', 'PopDens']

• Các tính năng được chọn để chạy Linear Regression:
  ['householdsize', 'racepctblack', 'racePctWhite', 'racePctAsian', 'racePctHisp', 'agePct12t29', 'agePct65up', 'numbUrban', 'pctUrban', 'pctWWage', 'pctWFarmSelf', 'pctWInvInc', 'pctWSocSec', 'pctWPubAsst', 'pctWRetire', '